In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window

### **Importing Prerequisites**

In [0]:
%run "/Workspace/Users/0bcr9999@gmail.com/FMGC-DATAENGINEERING-PROJECT/notebooks/setup/3. utilities"

In [0]:
dbutils.widgets.text("catalog", "fmcg")
dbutils.widgets.text("data_source", "", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

print(catalog, data_source)

### **Read Data from Silver Layer**

In [0]:
df_silver = spark.read.table(f"{catalog}.{silver_schema}.{data_source}")

In [0]:
df_gold = df_silver.select(
    F.col("order_id"),
    F.col("order_placement_date").alias("date"),
    F.col("customer_id").alias("customer_code"),
    F.col("product_code"),
    F.col("product_id"),
    F.col("order_qty").alias("sold_quantity")
)

In [0]:
display(df_gold.limit(5))

### **Write the Final Data Frame to Gold Schema**

In [0]:
if not (spark.catalog.tableExists("fmcg.gold.sb_factorders")):
    print("Creating New Table")
    df_gold.write \
        .format("delta") \
            .option("delta.enableChangeDataFeed", True) \
                .option("mergeSchema", True) \
                    .mode("overwrite") \
                        .saveAsTable(f"{catalog}.{gold_schema}.sb_fact{data_source}")

else:
    gold_delta_table = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.sb_fact{data_source}")
    
    gold_delta_table.alias("t").merge(df_gold.alias("s"), 
                                        "t.product_code == s.product_code AND  \
                                            t.order_id == s.order_id And \
                                                t.customer_code == s.customer_code AND\
                                                    t.date == s.date \
                                                    ").whenMatchedUpdateAll() \
                                                        .whenNotMatchedInsertAll() \
                                                            .execute()


### **Merging Child Company Data with Parent Company**

In [0]:
df_child = spark.sql(f"SELECT date, product_code, customer_code, sold_quantity from {catalog}.{gold_schema}.sb_fact{data_source}")

In [0]:
df_child.show(10)

In [0]:
df_child.count()

In [0]:
# Apply business logic of parent company to child for merging both.

# 1. Change the date to first day of month.
# 2025-07-15 -> 2025-07-01
# 2025-07-18 -> 2025-07-01

df_child_monthly = (
    df_child
    # Change the date to first day of month
    .withColumn("month_start", F.trunc("date", "MM")) 
    # Group monthly grain and then add sold_quantity
    .groupBy("month_start", "product_code", "customer_code")
    .agg(
        F.sum("sold_quantity").alias("sold_quantity")
    )
    # rename month_start back to date
    .withColumnRenamed("month_start", "date")
    )

df_child_monthly.show(50)

In [0]:
df_child_monthly.count()

In [0]:
gold_parent_delta = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.fact_{data_source}")

gold_parent_delta.alias("t").merge(df_child_monthly.alias("s"), 
                                    "t.product_code == s.product_code AND  \
                                        t.date == s.date AND \
                                            t.customer_code == s.customer_code \
                                                ").whenMatchedUpdateAll() \
                                                    .whenNotMatchedInsertAll() \
                                                        .execute()